In [2]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

# 1) Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2) Fit a CART regressor
tree = DecisionTreeRegressor(
    max_depth=3,         # control tree height
    min_samples_leaf=5,  # avoid tiny leaves
    random_state=42
)
tree.fit(X_train, y_train)

# -- FOR CLASSIFICATION --
# To build a classifier instead, swap the regressor:
# tree = DecisionTreeClassifier(
#     max_depth=4,
#     min_samples_split=10,
#     random_state=42
# )
# tree.fit(X_train, y_train)

# 3) Evaluate
train_mse = ((y_train - tree.predict(X_train))**2).mean()
test_mse  = ((y_test  - tree.predict(X_test))**2).mean()
print(f"Train MSE: {train_mse:.3f}, Test MSE: {test_mse:.3f}")

# -- FOR CLASSIFICATION --
# train_acc = tree.score(X_train, y_train)
# test_acc  = tree.score(X_test, y_test)
# print(f"Train Acc: {train_acc:.3f}, Test Acc: {test_acc:.3f}")

NameError: name 'X' is not defined

In [1]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Assume `tree` is your fitted DecisionTreeRegressor or DecisionTreeClassifier,
# and `feature_names` (and for classification, `class_names`) are defined.

# 1) Adjust figure size for large trees
plt.figure(figsize=(12, 8), dpi=100)

# 2) Call plot_tree with helpful options
plot_tree(
    tree,
    feature_names=feature_names,  # list of feature names
    class_names=class_names,      # only for classifier; omit for regressor
    filled=True,                  # color nodes by class/value
    rounded=True,                 # rounded box corners
    impurity=False,               # hide impurity (Gini/MSE) to reduce clutter
    proportion=True,              # show proportions instead of raw counts
    max_depth=4,                  # limit depth when tree is huge
    fontsize=8                     # shrink text for readability
)

plt.title("CART Tree")
plt.tight_layout()
plt.show()


NameError: name 'tree' is not defined

<Figure size 1200x800 with 0 Axes>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# 1) Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 2) Fit bagging ensemble with OOB scoring
bag = BaggingRegressor(
    base_estimator=DecisionTreeRegressor(),
    n_estimators=100,      # B
    oob_score=True,        # enable OOB
    bootstrap=True,        # sample with replacement
    random_state=42,
    n_jobs=-1
)
bag.fit(X_train, y_train)

# 3) Compute OOB error (MSE on samples not used in each tree)
oob_pred = bag.oob_prediction_
oob_mse  = mean_squared_error(y_train, oob_pred)

# 4) Compute ensemble variance on test set
#    Stack predictions from each tree (shape: n_estimators times n_test)
all_preds = np.stack([est.predict(X_test) for est in bag.estimators_], axis=0)
#    Variance at each test point, then average
ens_var = np.mean(np.var(all_preds, axis=0))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score

# 1) Split data as usual

# ----- Regression -----
rf_reg = RandomForestRegressor(
    n_estimators=100,       # number of trees B
    max_features='sqrt',    # m features per split
    min_samples_leaf=5,     # leaf size to control overfitting
    oob_score=True,         # enable out-of-bag estimation
    random_state=42
)
rf_reg.fit(X_train, y_train)

# Predictions & evaluation
y_pred = rf_reg.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

# ----- Classification -----
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    min_samples_split=10,
    oob_score=True,
    random_state=42
)
rf_clf.fit(X_train, y_train)

# Predict & evaluate
y_hat = rf_clf.predict(X_test)
acc = accuracy_score(y_test, y_hat)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# rf: fitted RandomForestRegressor or RandomForestClassifier
# feature_names: list of p feature labels

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(6,4))
plt.barh([feature_names[i] for i in indices],
         importances[indices])
plt.xlabel("Importance")
plt.title("Feature Importances\n(Mean Decrease in Impurity)")
plt.gca().invert_yaxis()  # highest at top
plt.tight_layout()


In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    rf, X_train, y_train,
    scoring='neg_mean_squared_error',  # or 'accuracy'
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)
perm_imp = result.importances_mean  # mean drop per feature


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, accuracy_score

# 1) Split data as usual

# ----- Regression -----
gb_reg = GradientBoostingRegressor(
    n_estimators=100,    # number of trees B
    learning_rate=0.1,   # shrinkage parameter (lambda)
    max_depth=1,         # depth of each tree
    subsample=1.0,       # fraction of samples per tree - if < 1 -> Stochastic Boosting
    random_state=42
)
gb_reg.fit(X_train, y_train)

# Predictions & evaluation
y_pred = gb_reg.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

# ----- Classification -----
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=1,
    subsample=1.0,
    random_state=42
)
gb_clf.fit(X_train, y_train)

# Predict & evaluate
y_hat = gb_clf.predict(X_test)
acc = accuracy_score(y_test, y_hat)


In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error, accuracy_score

# 1) Split data as usual

# ----- Regression -----
xgb_reg = XGBRegressor(
    n_estimators=100,      # number of boosting rounds (trees)
    learning_rate=0.1,     # step size shrinkage (lambda)
    gamma=0.0,             # Minimum loss reduction required to make a split
    max_depth=3,           # maximum depth of a tree
    subsample=1.0,         # fraction of rows per tree (for stochastic boosting)
    colsample_bytree=1.0,  # fraction of features per tree
    reg_lambda=1.0,        # L2 regularization term on weights
    reg_alpha=0.0,         # L1 regularization term on weights
    tree_method="hist",    # tree construction algorithm ("auto", "exact", "hist", "gpu_hist")
    random_state=42,
    n_jobs=-1,             # number of parallel threads (use all cores)
    verbosity=0            # 0 = silent, 1 = warning, 2 = info
)
xgb_reg.fit(X_train, y_train)

# Predictions & evaluation
y_pred = xgb_reg.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

# ----- Classification -----
xgb_clf = XGBClassifier(...)
xgb_clf.fit(X_train, y_train)

# Predict & evaluate
y_hat = xgb_clf.predict(X_test)
acc = accuracy_score(y_test, y_hat)


In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

# 1) Split data as usual
# 2) Define parameter grid to search
param_grid = {
    'max_depth': [3, 5, 7],      # try trees of depth 3, 5, 7
    'learning_rate': [0.01, 0.1], # step size shrinkage
    'n_estimators': [100, 200],   # number of trees
    'subsample': [0.8, 1.0]       # row subsampling for stochasticity
}
# 3) Create the XGBRegressor object
xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
# 4) Setup the grid search
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='neg_mean_squared_error', # minimize MSE
    cv=3,              # 3-fold cross-validation
    verbose=1,         # print progress
    n_jobs=-1          # parallelize
)
# 5) Fit grid search to the training data
grid.fit(X_train, y_train)
# 6) Retrieve the best estimator and evaluate
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("Best Params:", grid.best_params_)
print("Test MSE:", mse)
